In [13]:
# Зареждане на набора от данни
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]

In [18]:
# Конструиране на GraphSAGE модела
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)

        return x


In [19]:
# Създаване на модела
model = GraphSAGE(dataset.num_node_features, 64,
                  dataset.num_classes)


In [20]:
# Дефиниране на оптимизатор
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.005,
    weight_decay=5e-4)


In [21]:
# Функция на загубите
criterion = torch.nn.CrossEntropyLoss()


In [24]:
# Обучение на модела
num_epochs = 200

train_loss = []
train_acc = []
test_acc = []

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index)

    loss = criterion(out[data.train_mask],
        data.y[data.train_mask] )

    loss.backward()
    optimizer.step()
    train_loss.append(loss.item())
    model.eval()

    with torch.no_grad():
        pred = out.argmax(dim=1)
        train_correct = (pred[data.train_mask] == 
            data.y[data.train_mask]).sum()

        train_accuracy = (train_correct /
            data.train_mask.sum()).item()

        train_acc.append(train_accuracy)

        test_correct = (pred[data.test_mask] ==
            data.y[data.test_mask]).sum()

        test_accuracy = (test_correct /
            data.test_mask.sum()).item()

        test_acc.append(test_accuracy)


In [26]:
# Изчисляване на Accuracy
model.eval()
with torch.no_grad():
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

    correct = (pred[data.test_mask] ==
        data.y[data.test_mask]).sum()
    accuracy = (correct / data.test_mask.sum()).item()

print(f"Accuracy: {accuracy:.4f}")


Accuracy: 0.8060


In [27]:
# Изчисляване на F1-score
from sklearn.metrics import f1_score

model.eval()

with torch.no_grad():
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

f1 = f1_score(
    data.y[data.test_mask].cpu(),
    pred[data.test_mask].cpu(),
    average="macro")

print(f"F1-score: {f1:.4f}")


F1-score: 0.8011


In [28]:
# Брой параметри
num_params = sum(p.numel() for p in model.parameters())
print(num_params)

184391
